# 03 — 2D Parameter Sweep Visualizations

双参数扫描网格可视化：热图 / 等高线 / 3D 曲面 / Loss 轮廓。

顶部定义四个可复用函数，底部为调用示例。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — required for projection='3d'

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'

## 工具函数

In [ ]:
def plot_heatmap_from_data(data_file, output_dir=None):
    """
    读取 T_forward 结果并绘制热图（带数值标注）。

    Parameters
    ----------
    data_file : str — 制表符分隔的数据文件，含 r_11 / r_12 / T_forward 三列
    output_dir : str or None — 图片输出目录
    """
    # ---- 读取（容错多种编码）----
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except Exception:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')

    df.columns = df.columns.str.replace('\ufffd\ufffd', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11(μm)', 'r_12(μm)', 'T_forward']

    print("Columns:", df.columns.tolist())
    print("Shape:", df.shape)
    print(df.head())

    r_11_unique = sorted(df['r_11(μm)'].unique())
    r_12_unique = sorted(df['r_12(μm)'].unique())

    T_matrix = np.zeros((len(r_11_unique), len(r_12_unique)))
    for _, row in df.iterrows():
        i = r_11_unique.index(row['r_11(μm)'])
        j = r_12_unique.index(row['r_12(μm)'])
        T_matrix[i, j] = row['T_forward']

    # ---- 绘图 ----
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(T_matrix, cmap='viridis', aspect='auto', origin='lower',
                   vmin=np.min(T_matrix), vmax=np.max(T_matrix))

    ax.set_xticks(range(len(r_12_unique)))
    ax.set_xticklabels([f"{r:.1f}" for r in r_12_unique])
    ax.set_yticks(range(len(r_11_unique)))
    ax.set_yticklabels([f"{r:.1f}" for r in r_11_unique])
    ax.set_xlabel('r_12 (μm)', fontsize=14)
    ax.set_ylabel('r_11 (μm)', fontsize=14)
    ax.set_title('r_11 & r_12 vs Fundamental TE T_forward', fontsize=16, fontweight='bold')

    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('T_forward', fontsize=14)

    for i in range(len(r_11_unique)):
        for j in range(len(r_12_unique)):
            ax.text(j, i, f'{T_matrix[i, j]:.3f}', ha="center", va="center",
                    color='white', fontsize=8, fontweight='bold')

    ax.set_xticks(np.arange(len(r_12_unique)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(r_11_unique)) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=1)
    fig.tight_layout()

    # ---- 保存 ----
    if output_dir is None:
        output_dir = os.path.dirname(data_file) or '.'
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir,
                            f"{os.path.splitext(os.path.basename(data_file))[0]}_heatmap.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"Heatmap saved: {out_path}")
    plt.show()

    # ---- 统计 ----
    max_idx = np.unravel_index(np.argmax(T_matrix), T_matrix.shape)
    print(f"Max T_forward: {T_matrix[max_idx]:.6f}  @ r_11={r_11_unique[max_idx[0]]:.1f}, r_12={r_12_unique[max_idx[1]]:.1f}")
    print(f"Avg T_forward: {np.mean(T_matrix):.6f}")

    return T_matrix, r_11_unique, r_12_unique

In [ ]:
def create_contour_plot(data_file, output_dir=None):
    """等高线图（补充热图）"""
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except Exception:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')

    df.columns = df.columns.str.replace('\ufffd\ufffd', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11(μm)', 'r_12(μm)', 'T_forward']

    r_11_unique = sorted(df['r_11(μm)'].unique())
    r_12_unique = sorted(df['r_12(μm)'].unique())

    T_matrix = np.zeros((len(r_11_unique), len(r_12_unique)))
    for _, row in df.iterrows():
        i = r_11_unique.index(row['r_11(μm)'])
        j = r_12_unique.index(row['r_12(μm)'])
        T_matrix[i, j] = row['T_forward']

    R11, R12 = np.meshgrid(r_12_unique, r_11_unique)

    plt.figure(figsize=(10, 8))
    contour = plt.contour(R11, R12, T_matrix, levels=15, colors='black', linewidths=0.5)
    contourf = plt.contourf(R11, R12, T_matrix, levels=20, cmap='viridis', alpha=0.8)
    plt.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    plt.colorbar(contourf, label='T_forward')
    plt.xlabel('r_12 (μm)', fontsize=14)
    plt.ylabel('r_11 (μm)', fontsize=14)
    plt.title('T_forward Contour Plot', fontsize=16)
    plt.grid(True, alpha=0.3)

    if output_dir is None:
        output_dir = os.path.dirname(data_file) or '.'
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir,
                            f"{os.path.splitext(os.path.basename(data_file))[0]}_contour.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"Contour saved: {out_path}")
    plt.show()

In [ ]:
def plot_3d_surface_from_data(data_file, output_dir=None):
    """
    读取 T_forward 并绘制 3D 表面 + 2D 等高线投影。
    """
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except Exception:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')

    df.columns = df.columns.str.replace('\ufffd\ufffd', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11', 'r_12', 'T_forward']

    print("Columns:", df.columns.tolist(), "| Shape:", df.shape)
    print(f"r_11: {df['r_11'].min():.1f} – {df['r_11'].max():.1f}")
    print(f"r_12: {df['r_12'].min():.1f} – {df['r_12'].max():.1f}")
    print(f"T_forward: {df['T_forward'].min():.6f} – {df['T_forward'].max():.6f}")

    r_11 = df['r_11'].values
    r_12 = df['r_12'].values
    T_forward = df['T_forward'].values

    r_11_lin = np.linspace(r_11.min(), r_11.max(), 100)
    r_12_lin = np.linspace(r_12.min(), r_12.max(), 100)
    R12_grid, R11_grid = np.meshgrid(r_12_lin, r_11_lin)

    T_grid = griddata((r_12, r_11), T_forward, (R12_grid, R11_grid), method='cubic')
    mask = ~np.isnan(T_grid)
    if not np.all(mask):
        T_linear = griddata((r_12, r_11), T_forward, (R12_grid, R11_grid), method='linear')
        T_grid = np.where(mask, T_grid, T_linear)

    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(224)
    contour = ax.contour(R12_grid, R11_grid, T_grid, levels=15, colors='black', linewidths=0.5)
    contourf = ax.contourf(R12_grid, R11_grid, T_grid, levels=20, cmap='viridis', alpha=0.8)
    ax.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    ax.set_xlabel('r_12 (μm)', fontsize=12)
    ax.set_ylabel('r_11 (μm)', fontsize=12)
    ax.set_title('2D Contour Projection', fontsize=14)
    fig.colorbar(contourf, ax=ax, label='T_forward')
    fig.tight_layout()

    if output_dir is None:
        output_dir = os.path.dirname(data_file) or '.'
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir,
                            f"{os.path.splitext(os.path.basename(data_file))[0]}_3D_analysis.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"3D analysis saved: {out_path}")
    plt.show()
    return fig

In [ ]:
def plot_loss_contour_from_data(data_file, output_dir=None):
    """
    从 T_forward 计算 loss (dB) 并绘制 Loss 等高线图。
    """
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except Exception:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')

    df.columns = df.columns.str.replace('\ufffd\ufffd', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11', 'r_12', 'T_forward']

    r_11 = df['r_11'].values
    r_12 = df['r_12'].values
    T_forward = df['T_forward'].values

    r_11_lin = np.linspace(r_11.min(), r_11.max(), 100)
    r_12_lin = np.linspace(r_12.min(), r_12.max(), 100)
    R12_grid, R11_grid = np.meshgrid(r_12_lin, r_11_lin)

    T_grid = griddata((r_12, r_11), T_forward, (R12_grid, R11_grid), method='cubic')
    mask = ~np.isnan(T_grid)
    if not np.all(mask):
        T_linear = griddata((r_12, r_11), T_forward, (R12_grid, R11_grid), method='linear')
        T_grid = np.where(mask, T_grid, T_linear)

    loss = -10 * np.log10(T_grid)

    fig, ax = plt.subplots(figsize=(12, 10))
    levels = np.linspace(0.8, loss.max(), 30)
    contour = ax.contour(R12_grid, R11_grid, loss, levels=levels, colors='black', linewidths=0.5)
    contourf = ax.contourf(R12_grid, R11_grid, loss, levels=levels, cmap='viridis_r', alpha=0.8)
    ax.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    ax.set_xlabel(r'$\mathbf{r}_{\boldsymbol{12}}$ (μm)', fontsize=12, fontweight='bold')
    ax.set_ylabel(r'$\mathbf{r}_{\boldsymbol{11}}$ (μm)', fontsize=12, fontweight='bold')
    fig.tight_layout()

    if output_dir is None:
        output_dir = os.path.dirname(data_file) or '.'
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir,
                            f"{os.path.splitext(os.path.basename(data_file))[0]}_loss_contour.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    print(f"Loss contour saved: {out_path}")
    plt.show()
    return fig

## 调用示例

In [ ]:
# ---- 热图 ----
data_file = r"D:/simulation/Simulation Project/results/r_11_r_12_scan/T_forward_sweep_results.txt"
T_matrix, r11_vals, r12_vals = plot_heatmap_from_data(data_file)
# create_contour_plot(data_file)            # 可选：等高线
# plot_3d_surface_from_data(data_file)      # 可选：3D 曲面
# plot_loss_contour_from_data(data_file)    # 可选：Loss 轮廓

## d1 × d2 三维 Loss 曲面（多波长）

以下来自 LD-PWB-FA-bridge 的 Taper 参数扫描。

In [ ]:
# ---- @1550 nm ----
import glob

root_dir = r"D:\仿真\仿真数据\LD-PWB-FA-bridge\LD-PWB-Taper"
results = []

for d2_folder in glob.glob(os.path.join(root_dir, "step2*")):
    d2 = float(os.path.basename(d2_folder).split("_")[1])
    for txt_file in glob.glob(os.path.join(d2_folder, "*.txt")):
        d1 = float(os.path.basename(txt_file).replace(".txt", ""))
        data = np.loadtxt(txt_file, delimiter=",", skiprows=3)
        wavelength = data[:, 0] * 1e9
        t_forward = data[:, 1]
        idx = np.argmin(np.abs(wavelength - 1550))
        t_val = -np.log10(t_forward[idx]) * 10
        results.append({"d2": d2, "d1": d1, "t_1550": t_val})

df = pd.DataFrame(results)
d1 = df['d1'].values
d2 = df['d2'].values
T = df['t_1550'].values

d1_lin = np.linspace(d1.min(), d1.max(), 100)
d2_lin = np.linspace(d2.min(), d2.max(), 100)
D2_grid, D1_grid = np.meshgrid(d2_lin, d1_lin)
T_grid = griddata((d2, d1), T, (D2_grid, D1_grid), method='cubic')

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(D2_grid, D1_grid, T_grid, cmap='jet_r', edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.15)
ax.set_xlabel('d2 (μm)')
ax.set_ylabel('d1 (μm)')
ax.set_xlim(1.15, 0.65)
ax.set_ylim(1.9, 0.7)
ax.set_zlabel('TE Mode Loss (@1550 nm)')
ax.set_title(r'Transmission $\eta_1$ at 1550 nm')
ax.set_zlim(1.6, 1.0)
plt.show()

In [ ]:
# ---- @1540 nm ----
results = []
for d2_folder in glob.glob(os.path.join(root_dir, "step2*")):
    d2 = float(os.path.basename(d2_folder).split("_")[1])
    for txt_file in glob.glob(os.path.join(d2_folder, "*.txt")):
        d1 = float(os.path.basename(txt_file).replace(".txt", ""))
        data = np.loadtxt(txt_file, delimiter=",", skiprows=3)
        wavelength = data[:, 0] * 1e9
        t_forward = data[:, 1]
        idx = np.argmin(np.abs(wavelength - 1540))
        t_val = -np.log10(t_forward[idx]) * 10
        results.append({"d2": d2, "d1": d1, "t_1540": t_val})

df = pd.DataFrame(results)
d1 = df['d1'].values
d2 = df['d2'].values
T = df['t_1540'].values

d1_lin = np.linspace(d1.min(), d1.max(), 100)
d2_lin = np.linspace(d2.min(), d2.max(), 100)
D2_grid, D1_grid = np.meshgrid(d2_lin, d1_lin)
T_grid = griddata((d2, d1), T, (D2_grid, D1_grid), method='cubic')

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(D2_grid, D1_grid, T_grid, cmap='jet_r', edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.15)
ax.set_xlabel('d2 (μm)')
ax.set_ylabel('d1 (μm)')
ax.set_xlim(1.15, 0.65)
ax.set_ylim(1.9, 0.7)
ax.set_zlabel('TE Mode Loss (@1540 nm)')
ax.set_title(r'Transmission $\eta_1$ at 1540 nm')
ax.set_zlim(1.7, 1.0)
plt.show()

In [ ]:
# ---- @1560 nm ----
results = []
for d2_folder in glob.glob(os.path.join(root_dir, "step2*")):
    d2 = float(os.path.basename(d2_folder).split("_")[1])
    for txt_file in glob.glob(os.path.join(d2_folder, "*.txt")):
        d1 = float(os.path.basename(txt_file).replace(".txt", ""))
        data = np.loadtxt(txt_file, delimiter=",", skiprows=3)
        wavelength = data[:, 0] * 1e9
        t_forward = data[:, 1]
        idx = np.argmin(np.abs(wavelength - 1560))
        t_val = -np.log10(t_forward[idx]) * 10
        results.append({"d2": d2, "d1": d1, "t_1560": t_val})

df = pd.DataFrame(results)
d1 = df['d1'].values
d2 = df['d2'].values
T = df['t_1560'].values

d1_lin = np.linspace(d1.min(), d1.max(), 100)
d2_lin = np.linspace(d2.min(), d2.max(), 100)
D2_grid, D1_grid = np.meshgrid(d2_lin, d1_lin)
T_grid = griddata((d2, d1), T, (D2_grid, D1_grid), method='cubic')

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(D2_grid, D1_grid, T_grid, cmap='jet_r', edgecolor='none')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.15)
ax.set_xlabel('d2 (μm)')
ax.set_ylabel('d1 (μm)')
ax.set_xlim(1.15, 0.65)
ax.set_ylim(1.9, 0.7)
ax.set_zlabel('TE Mode Loss (@1560 nm)')
ax.set_title(r'Transmission $\eta_1$ at 1560 nm')
ax.set_zlim(1.6, 1.0)
plt.show()